In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import functions as f


# -------------------------------------------------------------------
# Base Transformer class
# This acts as an interface / contract for all transformer classes.
# Any class inheriting Transformer must implement the transform() method.
# -------------------------------------------------------------------
class Transformer:
    def transform(self, inputDFs):
        # Abstract method - must be implemented by subclasses
        raise NotImplementedError("Transform method must be implemented")


# -------------------------------------------------------------------
# FirstTransformer
# Purpose:
# Implements business logic to identify customers who generate
# the highest total revenue over their lifetime.
# -------------------------------------------------------------------
class CustomerRevenueOverTheLifetime(Transformer):

    def transform(self, inputDFs):
        """
        Business Question:
        Which customers generate the highest total revenue over their lifetime?

        Metrics calculated per customer:
        - Total number of orders
        - Total revenue generated
        - Average order value
        - First order date
        - Last order date
        """

        # ---------------------------------------------------------------
        # Step 1: Read Orders DataFrame from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')

        # ---------------------------------------------------------------
        # Step 2: Aggregate order-level data at customer level
        # - Count total orders per customer
        # - Calculate total and average revenue
        # - Identify first and last order dates
        # ---------------------------------------------------------------
        filterOrderDF = order_silverInputDeltaDF.groupBy('customer_id').agg(
            count('order_id').alias('Total_orders'),
            sum('total_amount').alias('Total_revenue'),
            avg('total_amount').alias('Average_order_value'),
            min('order_date').alias('First_order_date'),
            max('order_date').alias('Last_order_date')
        )

        # ---------------------------------------------------------------
        # Step 3: Read Customer Silver Delta DataFrame
        # Contains cleaned and validated customer master data
        # ---------------------------------------------------------------
        customer_silverInputDeltaDF = inputDFs.get('customer_silverInputDeltaDF')

        # ---------------------------------------------------------------
        # Step 4: Join customer master data with aggregated order metrics
        # Join Condition:
        # customer_id from customer table = customer_id from orders aggregation
        # Join Type: Inner (only customers with orders)
        # ---------------------------------------------------------------
        joinDf = customer_silverInputDeltaDF.alias('cs') \
            .join(
                filterOrderDF.alias('fo'),
                col('cs.customer_id') == col('fo.customer_id'),
                'inner'
            )

        # ---------------------------------------------------------------
        # Step 5: Select final business-friendly output columns
        # Rename full_name to customer_name for reporting clarity
        # ---------------------------------------------------------------
        return joinDf.select(
            col('cs.customer_id'),
            col('cs.full_name').alias('customer_name'),
            col('fo.Total_orders'),
            col('fo.Total_revenue'),
            col('fo.Average_order_value'),
            col('fo.First_order_date'),
            col('fo.Last_order_date')
        )

# -------------------------------------------------------------------
# DailyRevenueTrend Transformer
# Purpose:
# Calculates daily revenue trends excluding cancelled orders
# -------------------------------------------------------------------
class DailyRevenueTrend(Transformer):

    def transform(self, inputDFs):
        """
        Business Question:Daily Revenue Trend
        
        How does daily revenue change over time?

        Requirements : Revenue per day, Order count per day, Exclude cancelled orders

        """
        # ---------------------------------------------------------------
        # Step 1: Read Orders DataFrame from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')

        # ---------------------------------------------------------------
        # Step 2: Filter out cancelled orders and aggregate by order_date
        # - Sum total revenue per day
        # - Count number of orders per day
        # ---------------------------------------------------------------
        filterOrderDF = (order_silverInputDeltaDF.filter(col("order_status") != 'cancelled').groupBy(
            'order_date'
        ).agg(
            sum('total_amount').alias('Revenue_per_day'),
            count('order_id').alias('Order_count_per_day')
           )
        )
    
        # ---------------------------------------------------------------
        # Step 3: Select final columns for daily revenue trend output
        # ---------------------------------------------------------------
        return filterOrderDF.select(
            'order_date',
            'Revenue_per_day',
            'Order_count_per_day'

        )

# -------------------------------------------------------------------
# TopCitiesByRevenue Transformer
# Purpose:
# Identifies cities that contribute the most revenue
# -------------------------------------------------------------------        
class TopCitiesByRevenue:

    """
        Business question
        Top Cities by Revenue Which cities contribute the most revenue? 
        Requirements : Total revenue per city, Number of unique customers, Average order value
        """
        
    def transform(self, inputDFs):
        # ---------------------------------------------------------------
        # Step 1: Extract required DataFrames from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')
        customer_silverInputDeltaDF = inputDFs.get('customer_silverInputDeltaDF')
    
        # ---------------------------------------------------------------
        # Step 2: Filter orders to include only delivered orders
        # ---------------------------------------------------------------
        filterOrderDF = order_silverInputDeltaDF.filter(col("order_status").isin(['Delivered']))

        # filterOrderDF.show()        

        # ---------------------------------------------------------------
        # Step 3: Join orders with customer data to get city information
        # Join Type: Inner (only customers with delivered orders)
        # ---------------------------------------------------------------
        joinDF = filterOrderDF.alias('o').join(
        customer_silverInputDeltaDF.alias('c'),
        col('o.customer_id') == col('c.customer_id'),'inner')

        # ---------------------------------------------------------------
        # Step 4: Aggregate metrics by city
        # - Total revenue per city
        # - Count of unique customers per city
        # - Average order value per city (rounded to 2 decimals)
        # ---------------------------------------------------------------
        resultDF = joinDF.groupBy(col('c.city')).agg(
        sum('o.total_amount').alias('Total_revenue'),
        countDistinct('c.customer_id').alias('Unique_customers'),
        round(avg('o.total_amount'),2).alias('Average_order_value'))

        # ---------------------------------------------------------------
        # Step 5: Select final columns for city revenue analysis output
        # ---------------------------------------------------------------
        return resultDF.select(
            'city',
            'Total_revenue',
            'Unique_customers',
            'Average_order_value'
            )
        

# -------------------------------------------------------------------
# Repeat vs One-Time Customers Transformer
# Purpose:
# Identifies How many customers are repeat buyers vs one-time buyers?
# -------------------------------------------------------------------   

class repeatvsOneTimeCustomers:
    """
    Business question
    How many customers are repeat buyers vs one-time buyers?
    Requirements : Count customers with 1 order, Count customers with >1 order, Revenue contribution by each group
    """
    def transform(self,inputDFs):
        # ---------------------------------------------------------------
        # Step 1: Extract required DataFrames from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')

        # Step 2: Count orders per customer

        customer_order_count = order_silverInputDeltaDF.groupBy('customer_id').agg(
            count('order_id').alias('Total_order'),
            sum('total_amount').alias('lifetime_revenue')
            
        )
        # Step 3: Segment customers
        segemented = customer_order_count.withColumn(
            'customer_Segment',
            when(col('Total_order') == 1,'One-time customer')
            .otherwise('Repeat customer')
        )

        # Step 3: Aggregate by segment
        result = segemented.groupBy('customer_segment').agg(
                countDistinct('customer_id').alias('customer_count'),
                sum('lifetime_revenue').alias('total_revenue'),
                avg('lifetime_revenue').alias('avg_customer_lifetime_value')
            )
        
        return result.select(
            'customer_segment',
            'customer_count',
            'total_revenue',
            'avg_customer_lifetime_value'
        )

# -------------------------------------------------------------------
# Percentage of orders are delivered vs cancelled?
# Purpose:
# Identifies What percentage of orders are delivered vs cancelled?

# -------------------------------------------------------------------   

class orderDeliveredVsCancelled:

    """
    Business question : What percentage of orders are delivered vs cancelled?
    Order count by status, Percentage share, Daily breakdown
    """

    def transform(self,inputDFs):
        # ---------------------------------------------------------------
        # Step 1: Extract required DataFrames from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')

        # Step 2: Filter for delivered and cancelled orders only

        filter_df = order_silverInputDeltaDF.filter(
            f.col('order_status').isin('Delivered','Cancelled')
        )
        
        # Step 3: Count by status and date for daily breakdown
        daily_status_count = filter_df.groupBy('order_date','order_status')\
            .agg(f.count('order_id').alias('status_count'))

        # Step 4: Calculate percentage share per day
        win = Window.partitionBy('order_date')
        
        result = daily_status_count.withColumn(
            'Percentage_Share',
            round((f.col('status_count') / f.sum('status_count').over(win)) * 100,2)
        )

        return result.select(
            'order_date',
            'order_status',
            'status_count',
            'Percentage_Share'
        ).orderBy('order_date', 'order_status')


# -------------------------------------------------------------------
# Which payment methods drive the most revenue?
# Purpose:
# Identifies Which payment methods drive the most revenue?
# -------------------------------------------------------------------  

class paymentMethodRevenue(Transformer):
    """ 
    Business question
    Which payment methods drive the most revenue?
    Total revenue by payment method and Cancelled orders excluded
    """
    def transform(self,inputDFs):
        # ---------------------------------------------------------------
        # Step 1: Extract required DataFrames from input dictionary
        # ---------------------------------------------------------------

        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')

        # Step 2: Filter out cancelled orders

        filter_df = order_silverInputDeltaDF.filter(col('order_status') != 'Cancelled').groupBy('payment_method').agg(count('order_id').alias('Order_count_Per_Method'),
            sum('total_amount').alias('Total_revenue_Per_Method'))
        
        # ---------------------------------------------------------------
        # Step 3: Select final columns for most revenue per payment method output
        # ---------------------------------------------------------------
        
        return filter_df.select(
            'payment_method',
            'Order_count_Per_Method',
            'Total_revenue_Per_Method'
        )

class productCategoriesRevenue(Transformer):

    """
    Business question : Which product categories generate the most revenue?
    Revenue by category & sub-category, Total quantity sold, Top 5 products per category
    """
    def transform(self,inputDFs):
        # ---------------------------------------------------------------
        # Step 1: Extract required DataFrames from input dictionary
        # ---------------------------------------------------------------
        order_silverInputDeltaDF = inputDFs.get('order_silverInputDeltaDF')
        order_items_silverInputDeltaDF = inputDFs.get('order_items_silverInputDeltaDF')
        product_items_silverInputDeltaDF = inputDFs.get('product_items_silverInputDeltaDF')

        # Step 2: Join DF
        join_Order_Order_items = order_silverInputDeltaDF.alias('o').join(
            order_items_silverInputDeltaDF.alias('oi'),how = 'inner',
            on=col('o.order_id') == col('oi.order_id'))
        
        join_All = join_Order_Order_items.join(
            product_items_silverInputDeltaDF.alias('pi'),
            how = 'inner',
            on=col('oi.product_id') == col('pi.product_id')
        )

        # Step 3 : Revenue by category & sub-category, Total quantity sold
        # Filter Null 
        filter_Unknown = join_All.filter(col('pi.category')!= 'Unknown')
        revenue_summary = filter_Unknown.groupBy(
            col('pi.category'),
            col('pi.sub_category')
            ).agg(
            sum(col('oi.net_amount')).alias('total_revenue'),
            sum(col('oi.quantity')).alias('total_quantity_sold'),
            countDistinct(col('pi.product_id')).alias('product_count')
        ).orderBy(col('total_revenue'))

        product_revenue = filter_Unknown.groupBy(col('pi.category'),col('pi.product_id')).agg(
        sum(col('oi.net_amount')).alias('product_revenue'),
        sum(col('oi.quantity')).alias('product_quantity_sold')
        )
        win = Window.partitionBy('category').orderBy(col('product_revenue').desc())
        top_5_product = (
                        product_revenue.withColumn('top_product',row_number().over(win)).filter(col('top_product') <=5)
                        )
        
        return revenue_summary.select('category','sub_category','total_revenue','total_quantity_sold'),\
        top_5_product.select('category','product_id','product_revenue','top_product')




                                                               








        

       